# Phase 1 — Loading and Wrangling the PHEME Dataset

**Project:** Comparative spread analysis of rumors vs. facts on Twitter during breaking-news events.

**Goal of this notebook:** Walk the PHEME folder structure once, parse every cascade, and produce two clean, reusable artifacts:

1. `cascades_df` — one row per source tweet (the root of a conversation). Contains the veracity label, event, source-author features, and aggregate cascade stats we'll compute in Phase 2.
2. `reactions_df` — one row per reply tweet. Lets us reconstruct timing and authorship for any cascade.

We also keep the raw reply-tree structure as a dict keyed by source-tweet ID, because we'll need the actual tree topology (not just aggregate counts) for the network-structure metrics in Phase 2.

**Why this design:** keeping cascade-level rows separate from reaction-level rows is the standard pattern for cascade analysis (Vosoughi et al. 2018, Goel et al. 2016). It avoids row-explosion in the main dataframe and lets us join in the other direction only when needed.

**Folder layout we expect:**
```
<DATA_ROOT>/
  <event>-all-rnr-threads/
    rumours/<source_tweet_id>/
      source-tweets/<source_tweet_id>.json
      reactions/<reply_id>.json   (many)
      structure.json
      annotation.json
    non-rumours/<source_tweet_id>/
      ... same as above
```

## 0. Setup

In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from typing import Any

import pandas as pd

# ----- POINT THIS AT YOUR DATA -----
# Should be the folder that directly contains the *-all-rnr-threads subfolders.
# Example on your machine:
#   DATA_ROOT = Path('/path/to/rumors/all-rnr-annotated-threads')
DATA_ROOT = Path('/home/claude/pheme_sample')   # <-- replace with your local path

assert DATA_ROOT.exists(), f'DATA_ROOT does not exist: {DATA_ROOT}'
print('Using data root:', DATA_ROOT)

## 1. Helpers

Three small, focused functions:

- `parse_twitter_time` — Twitter's `created_at` is a non-ISO format (`'Wed Jan 07 11:11:33 +0000 2015'`). Parsing once here avoids subtle bugs later.
- `extract_user_features` — pulls the handful of user fields we actually care about (follower count, account age, verified flag) out of the deeply nested user object so the dataframe stays flat.
- `flatten_structure` — `structure.json` is a recursive nested dict. We need both (a) the parent→children edges and (b) the depth of every node. One pass gives us both.

In [ ]:
TWITTER_TIME_FMT = '%a %b %d %H:%M:%S %z %Y'

def parse_twitter_time(s: str) -> datetime:
    """Parse Twitter's 'created_at' field into an aware UTC datetime."""
    return datetime.strptime(s, TWITTER_TIME_FMT).astimezone(timezone.utc)


def extract_user_features(user: dict) -> dict:
    """Pull the user-level features that matter for spread analysis.

    We keep only what we'll plausibly use in Phase 3:
    - followers_count: a strong proxy for potential reach
    - friends_count, statuses_count, listed_count: activity / influence proxies
    - verified: known to correlate with reach during breaking news
    - account_created_at: lets us compute account age (anti-bot signal)
    """
    if not user:
        return {}
    return {
        'user_id': user.get('id_str'),
        'screen_name': user.get('screen_name'),
        'followers_count': user.get('followers_count'),
        'friends_count': user.get('friends_count'),
        'statuses_count': user.get('statuses_count'),
        'listed_count': user.get('listed_count'),
        'verified': user.get('verified'),
        'account_created_at': (
            parse_twitter_time(user['created_at']) if user.get('created_at') else None
        ),
    }


def flatten_structure(struct: dict, root_id: str) -> tuple[list[tuple[str, str]], dict[str, int]]:
    """Walk the recursive structure.json dict.

    Returns:
        edges: list of (parent_id, child_id) tuples
        depth: dict mapping every tweet_id (incl. root) to its depth (root = 0)

    Note on the format: PHEME's structure.json represents leaves as either an
    empty list `[]` or an empty dict `{}`. Both are handled.
    """
    edges: list[tuple[str, str]] = []
    depth: dict[str, int] = {root_id: 0}

    def walk(node: Any, parent_id: str, current_depth: int) -> None:
        if not isinstance(node, dict):
            return  # leaf (empty list or empty dict in PHEME)
        for child_id, sub in node.items():
            edges.append((parent_id, child_id))
            depth[child_id] = current_depth + 1
            walk(sub, child_id, current_depth + 1)

    # The top of structure.json is { root_id: { ...children... } }
    inner = struct.get(root_id, {})
    walk(inner, root_id, 0)
    return edges, depth

## 2. Per-cascade loader

A cascade = one source tweet + all its reactions + the reply-tree structure + the annotation. We package the parsed result into a small dataclass so downstream code is type-safe and self-documenting.

In [ ]:
@dataclass
class Cascade:
    source_id: str
    event: str
    is_rumour: bool
    veracity: str | None  # 'true' / 'false' / 'unverified' / None for non-rumours
    source_row: dict      # one row destined for cascades_df
    reaction_rows: list[dict]  # rows destined for reactions_df
    edges: list[tuple[str, str]]  # the reply tree as edge list
    depth: dict[str, int]         # tweet_id -> depth in tree


def load_cascade(cascade_dir: Path, event: str, is_rumour: bool) -> Cascade | None:
    """Parse one cascade folder. Returns None if essential files are missing.

    A few PHEME folders are known to be slightly corrupt (missing source tweet
    or annotation). We skip those rather than fail the whole pipeline.
    """
    source_id = cascade_dir.name
    src_file = cascade_dir / 'source-tweets' / f'{source_id}.json'
    struct_file = cascade_dir / 'structure.json'
    annot_file = cascade_dir / 'annotation.json'
    reactions_dir = cascade_dir / 'reactions'

    if not src_file.exists() or not struct_file.exists() or not annot_file.exists():
        return None

    with open(src_file) as f:
        src = json.load(f)
    with open(struct_file) as f:
        struct = json.load(f)
    with open(annot_file) as f:
        annot = json.load(f)

    # Veracity is only present for rumours, and the field is sometimes called
    # 'true' (a 0/1 flag in PHEME-9) or sometimes 'misinformation' depending on
    # the event. We collapse this to a single string label.
    veracity = None
    if is_rumour:
        if 'true' in annot:
            v = annot['true']
            veracity = {'1': 'true', '0': 'false', 1: 'true', 0: 'false'}.get(v, str(v))
        elif 'misinformation' in annot:
            v = annot['misinformation']
            veracity = {'1': 'false', '0': 'true', 1: 'false', 0: 'true'}.get(v, str(v))
        else:
            veracity = 'unverified'

    # ----- Source tweet row -----
    src_user = extract_user_features(src.get('user', {}))
    source_created_at = parse_twitter_time(src['created_at'])
    source_row = {
        'source_id': source_id,
        'event': event,
        'is_rumour': is_rumour,
        'veracity': veracity,
        'text': src.get('text'),
        'lang': src.get('lang'),
        'source_created_at': source_created_at,
        'retweet_count_snapshot': src.get('retweet_count'),  # at collection time
        'favorite_count_snapshot': src.get('favorite_count'),
        # source-author features (prefixed for clarity downstream):
        **{f'src_{k}': v for k, v in src_user.items()},
    }

    # ----- Reaction rows -----
    reaction_rows: list[dict] = []
    if reactions_dir.exists():
        for rf in reactions_dir.glob('*.json'):
            try:
                with open(rf) as f:
                    r = json.load(f)
            except json.JSONDecodeError:
                continue  # rare malformed file; skip
            r_user = extract_user_features(r.get('user', {}))
            reaction_rows.append({
                'source_id': source_id,
                'reply_id': r.get('id_str') or rf.stem,
                'in_reply_to_status_id': (
                    str(r['in_reply_to_status_id']) if r.get('in_reply_to_status_id') else None
                ),
                'reply_created_at': parse_twitter_time(r['created_at']) if r.get('created_at') else None,
                'text': r.get('text'),
                **{f'rep_{k}': v for k, v in r_user.items()},
            })

    # ----- Tree structure -----
    edges, depth = flatten_structure(struct, source_id)

    return Cascade(
        source_id=source_id,
        event=event,
        is_rumour=is_rumour,
        veracity=veracity,
        source_row=source_row,
        reaction_rows=reaction_rows,
        edges=edges,
        depth=depth,
    )

## 3. Walk the whole dataset

One pass over `<DATA_ROOT>/<event>-all-rnr-threads/{rumours,non-rumours}/<source_id>/`.

In [ ]:
def discover_events(root: Path) -> list[tuple[str, Path]]:
    """Find all '<event>-all-rnr-threads' folders under root. Returns [(event_name, path), ...]."""
    out = []
    for p in sorted(root.iterdir()):
        if p.is_dir() and p.name.endswith('-all-rnr-threads'):
            event = p.name.replace('-all-rnr-threads', '')
            out.append((event, p))
    # Fallback for the case where DATA_ROOT *is* a single event folder
    if not out and root.name.endswith('-all-rnr-threads'):
        out.append((root.name.replace('-all-rnr-threads', ''), root))
    return out


def load_all(root: Path, verbose: bool = True) -> list[Cascade]:
    cascades: list[Cascade] = []
    skipped = 0
    events = discover_events(root)
    if not events:
        raise RuntimeError(f'No *-all-rnr-threads folders found under {root}')

    for event, event_dir in events:
        for label_dir, is_rumour in [('rumours', True), ('non-rumours', False)]:
            sub = event_dir / label_dir
            if not sub.exists():
                continue
            for cascade_dir in sub.iterdir():
                if not cascade_dir.is_dir():
                    continue
                c = load_cascade(cascade_dir, event=event, is_rumour=is_rumour)
                if c is None:
                    skipped += 1
                else:
                    cascades.append(c)
        if verbose:
            n_event = sum(1 for c in cascades if c.event == event)
            print(f'  {event}: {n_event} cascades loaded')

    if verbose:
        print(f'\nTotal: {len(cascades)} cascades loaded, {skipped} skipped')
    return cascades


cascades = load_all(DATA_ROOT)

## 4. Build the two dataframes

We also stash the per-cascade tree structures (`edges`, `depth`) in a dict, ready for Phase 2.

In [ ]:
cascades_df = pd.DataFrame([c.source_row for c in cascades])
reactions_df = pd.DataFrame([row for c in cascades for row in c.reaction_rows])

# Tree structures, keyed by source_id. We'll use this in Phase 2 (NetworkX).
trees = {c.source_id: {'edges': c.edges, 'depth': c.depth} for c in cascades}

# Add a few useful derived columns now (cheap to compute, used everywhere downstream).
cascades_df['n_reactions'] = cascades_df['source_id'].map(
    reactions_df.groupby('source_id').size()
).fillna(0).astype(int)

cascades_df['max_depth'] = cascades_df['source_id'].map(
    {sid: max(t['depth'].values()) if t['depth'] else 0 for sid, t in trees.items()}
)

# Categorical label combining is_rumour and veracity, useful for grouped plots.
def _label(row):
    if not row['is_rumour']:
        return 'non-rumour'
    return f"rumour ({row['veracity']})" if row['veracity'] else 'rumour'
cascades_df['label'] = cascades_df.apply(_label, axis=1)

print('cascades_df shape: ', cascades_df.shape)
print('reactions_df shape:', reactions_df.shape)
cascades_df.head()

## 5. Sanity checks

Before we trust this data, run the cheap, obvious checks. **Catch problems here, not in Phase 3 when you're trying to interpret a finding.**

In [ ]:
print('--- Class balance overall ---')
print(cascades_df['label'].value_counts(dropna=False))

print('\n--- Class balance per event ---')
print(
    cascades_df.groupby('event')['label']
    .value_counts()
    .unstack(fill_value=0)
)

print('\n--- Cascade size distribution (n_reactions) ---')
print(cascades_df['n_reactions'].describe(percentiles=[.5, .9, .99]))

print('\n--- Tree depth distribution ---')
print(cascades_df['max_depth'].describe(percentiles=[.5, .9, .99]))

# Reactions referencing source_ids we don't have (data integrity check)
orphans = reactions_df.loc[~reactions_df['source_id'].isin(cascades_df['source_id'])]
print(f'\nOrphan reactions (no matching source): {len(orphans)}')

# Reply tweets with timestamps before their source tweet (would break time-to-X metrics)
ts_check = reactions_df.merge(
    cascades_df[['source_id', 'source_created_at']], on='source_id', how='left'
)
bad_ts = ts_check.loc[
    ts_check['reply_created_at'].notna()
    & (ts_check['reply_created_at'] < ts_check['source_created_at'])
]
print(f'Reactions with timestamp BEFORE source: {len(bad_ts)} (should be 0)')

## 6. A peek at one cascade end-to-end

Useful for the presentation later — we'll want to show one concrete example before showing aggregate plots.

In [ ]:
# Pick the largest cascade as an illustrative example.
demo = cascades_df.sort_values('n_reactions', ascending=False).iloc[0]
print(f"Source ID: {demo['source_id']}")
print(f"Event: {demo['event']}  |  Label: {demo['label']}")
print(f"Author: @{demo['src_screen_name']} ({demo['src_followers_count']:,} followers, verified={demo['src_verified']})")
print(f"Posted: {demo['source_created_at']}")
print(f"Text: {demo['text']}")
print(f"Cascade size: {demo['n_reactions']} reactions, max depth {demo['max_depth']}")

demo_replies = reactions_df.loc[reactions_df['source_id'] == demo['source_id']].sort_values('reply_created_at')
demo_replies.head(5)[['reply_id', 'reply_created_at', 'rep_screen_name', 'text']]

## 7. Persist for downstream phases

We save in Parquet (preserves dtypes; way faster than CSV) and pickle the trees dict.
Phase 2 will load these without re-walking thousands of folders.

In [ ]:
import pickle

OUT = Path('./processed')
OUT.mkdir(exist_ok=True)

cascades_df.to_parquet(OUT / 'cascades.parquet', index=False)
reactions_df.to_parquet(OUT / 'reactions.parquet', index=False)
with open(OUT / 'trees.pkl', 'wb') as f:
    pickle.dump(trees, f)

print(f'Wrote {OUT.resolve()}')
for p in OUT.iterdir():
    print(f'  {p.name}: {p.stat().st_size / 1024:.1f} KB')

---

## What's next (Phase 2 preview)

With `cascades_df`, `reactions_df`, and `trees` in hand, Phase 2 will compute the cascade-level metrics that drive the whole story:

- **Speed** — time-to-first-reply, time to reach 50% of final reactions (cascade half-life), reply velocity in the first hour.
- **Reach** — final size, unique users involved, number of distinct branches off the root.
- **Network shape** — max depth (already computed), max breadth at any level, and structural virality (Goel et al.). Structural virality is the key metric: it distinguishes "one big account broadcast it" from "it spread peer-to-peer". We'll use NetworkX for these.

Each metric becomes a new column on `cascades_df`. Then Phase 3 just compares distributions across `label`.